<a href="https://colab.research.google.com/github/smduarte/spbd-2526/blob/main/docs/labs/lab7/SPBD_Labs_spark4_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Spark Structured Streaming Exercises


##1. Weblog Analysis

Consider a stream of logging *events* for the web accesses.

Each logging event contain ***json*** lines as shown below:

```json
{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}```

In [ ]:
#@title Start the Structured Source

!wget -q -O - https://github.com/smduarte/spbd-2526/raw/main/docs/labs/lab7/json_logsender.tgz | tar xfz - 2> /dev/null

!nohup python json_logsender/server.py json_logsender/web.log 8888 > /dev/null 2> /dev/null &

In [ ]:
#@title Structured Streaming Session Example

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777

rawlines = spark.readStream.format("socket") \
    .option("host", "localhost") \
    .option("port", 8888) \
    .load()

query = rawlines \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='1 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

query.awaitTermination(20)
query.stop()

In [ ]:
#@title Structured Streaming Session Example

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()


# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

rawlines = spark.readStream.format("socket") \
    .option("host", "localhost") \
    .option("port", 8888) \
    .load()

json_lines = rawlines.select(from_json(col("value"), inferred_schema).alias("data")) \
 .select("data.*")

query = json_lines \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='1 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

query.awaitTermination(20)
query.stop()

# Exercises

***Every 3 seconds***,

1. Dump the number of requests in the last 10 seconds;
2. Dump the number of requests in the last 10 seconds, only if they total more than 100;
3. Dump the number of requests in the last 10 seconds, if there is an IP address with more than 100 requests;
4. Dump the proportion of IPv4 vs IPv6 requests in the last 20 seconds.
